In [1]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt
from itertools import chain, combinations

In [2]:
def powerset(iterable):
    "powerset([1,2,3]) --> () (1,) (2,) (3,) (1,2) (1,3) (2,3) (1,2,3)"
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s)+1))

def h_1(x,n):
    y = 1-(1-x)**n
    return(y)

def h_2(x,r):
    y = (1+r)*x-r*x**2
    return(y)

def h_3(x,alfa):
    return(min(1,x/(1-alfa)))

def h_4(x,r):
    if x < 1/2:
        return((1+r)*x)
    else:
        return((1-r)*p+r)

In [3]:
def Robust_exp (x,p,r,m):
    N = len(x)
    q = cp.Variable(N)
    q_b = cp.Variable(N)
    phi_cons = 0
    psets = list(powerset(list(range(N))))
    for i in range(1,len(psets)):
        psets[i] = list(psets[i])
    psets = psets[1:(len(psets)-1)]
    for i in range(N):
        phi_cons = phi_cons -(cp.entr(q[i]) + q[i]*np.log(p[i]))
    constraints = [q >= 0, q_b >= 0, cp.sum(q) == 1, cp.sum(q_b) == 1, phi_cons <= r]
    for i in range(1,len(psets)):
        z1 = q[psets[i]]
        z2 = q_b[psets[i]]
        constraints.append(cp.sum(z2)-(1-(1-cp.sum(z1))**m) <= 0)
    obj = cp.Minimize(q_b.T @ x)
    prob = cp.Problem(obj,constraints)
    prob.solve()
    return(prob.value,q.value,q_b.value)

In [4]:
N=4
x=np.array([1,2,3])
psets = list(powerset(list(range(N))))
for i in range(1,len(psets)):
    psets[i] = list(psets[i])
psets = psets[1:(len(psets)-1)]
psets

[[0],
 [1],
 [2],
 [3],
 [0, 1],
 [0, 2],
 [0, 3],
 [1, 2],
 [1, 3],
 [2, 3],
 [0, 1, 2],
 [0, 1, 3],
 [0, 2, 3],
 [1, 2, 3]]

In [5]:
x=np.array([1,2,3,4])
p=np.array([0.1,0.3,0.4,0.2])
r=1
m=4
[pvalue,qvalue,qbvalue]=Robust_exp(x,p,r,m)

In [7]:
print(pvalue)
print(qvalue)

1.0041618257560985
[0.68463718 0.14079564 0.09626113 0.07830605]
